# Glyph — Instruction-tuning LoRA (E5_instruct)

Обучает LoRA-адаптеры на synthetic датасете (этап B), используя формат `Вопрос→Ответ` с маскированием префикса — loss считается только на токенах ответа.

**Гиперпараметры:** r=16, alpha=32, lr=2e-4, 5 эпох (копия лучшей конфигурации из этапа 5).

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`.

## Что делает
1. Клонирует репо (должен содержать `ml/data/synthetic/*.jsonl` — закоммить после генерации)
2. (Альтернатива) Загружает zip с synthetic датасетом вручную
3. Устанавливает зависимости
4. Обучает 3 адаптера `E5_instruct` (Достоевский, Чехов, Булгаков)
5. Архивирует веса и даёт скачать

**Время на T4:** ~15-20 мин на автора × 3 = **~45-60 минут**.

## 1. Проверка GPU

In [ ]:
!nvidia-smi | head -15
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Клонирование репозитория

In [ ]:
%cd /content
!rm -rf glyph
!git clone --depth 1 https://github.com/tearfu1/glyph.git
%cd /content/glyph/ml

## 3. Проверка synthetic датасета

Если датасет уже закоммичен в репозиторий — пропусти ячейку «Альтернатива». Иначе — запусти её и загрузи zip с Colab B-ноутбука.

In [ ]:
import os

need_upload = True
for author in ['dostoevsky', 'chekhov', 'bulgakov']:
    path = f'data/synthetic/{author}.jsonl'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'  {author}: {size} bytes')
        need_upload = False
    else:
        print(f'  {author}: MISSING')

if need_upload:
    print('\nДатасет не найден в репо — запусти ячейку «Альтернатива».')

### Альтернатива: upload zip вручную (если synthetic не в репо)

Запусти только если датасет не закоммичен. Откроется диалог — выбери `glyph_synthetic_dataset.zip`, скачанный из ноутбука B.

In [ ]:
from google.colab import files
uploaded = files.upload()
!unzip -o glyph_synthetic_dataset.zip -d data/
!ls -la data/synthetic/

## 4. Установка зависимостей

В Colab уже есть torch+CUDA; ставим peft/datasets/accelerate.

In [ ]:
!pip install -q transformers==4.51.0 peft==0.18.1 datasets==2.19.0 accelerate==0.30.0

## 5. Быстрый тест (опционально): 1 эпоха на Булгакове

Булгаков — самый маленький корпус, быстро убедимся что скрипт вообще обучается.

**Пропусти эту ячейку**, если уверен — сразу к пункту 6.

In [ ]:
# Тестовый прогон 1 эпоха
!python -c "\
import sys; sys.path.insert(0, '.');\
from scripts import train_instruct_lora as m;\
m.LORA_PARAMS['epochs'] = 1;\
m.main()" --author bulgakov

## 6. Полное обучение для всех 3 авторов

5 эпох × 3 автора на T4 ≈ 45-60 минут.

In [ ]:
!python -m scripts.train_instruct_lora

## 7. Очистка trainer checkpoints (чтобы архив был маленький)

In [ ]:
!find models/adapters/*/E5_instruct -type d -name 'checkpoint-*' -exec rm -rf {} + 2>/dev/null
!ls -la models/adapters/dostoevsky/E5_instruct/
print('\nРазмеры адаптеров:')
!du -sh models/adapters/*/E5_instruct/

## 8. Скачать архив с адаптерами

In [ ]:
!cd models/adapters && zip -r /content/glyph_e5_instruct_adapters.zip \
    dostoevsky/E5_instruct chekhov/E5_instruct bulgakov/E5_instruct

!cp models/experiment_results_instruct.jsonl /content/ 2>/dev/null || true

from google.colab import files
files.download('/content/glyph_e5_instruct_adapters.zip')